In [ ]:
import os

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
from typing import Annotated, List

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
# from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI


class State(TypedDict):
    chunks: List[str]  # List of chunks to classify
    legal_chunks: List[str] # List of legal chunks

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")


def classify_chunks(state: State):
    legal_chunks = []
    for chunk in state["chunks"]:
        prompt = [
            SystemMessage(
                content="""You are a strict legal document classifier. 
                Determine if the following text chunk contains explicit legal terms, contractual clauses, or references to laws. 
                Answer ONLY 'YES' if the chunk is exclusively legal, and 'NO' if it is not.
                Examples:
                'The contract is valid' - YES
                'The weather is nice' - NO
                'Article 12 of the agreement' - YES
                'I ate lunch' - NO
                """
            ),
            HumanMessage(content=chunk),
        ]
        response = llm.invoke(prompt).content.lower()
        if "yes" in response:
            legal_chunks.append(chunk)

    return {"legal_chunks": legal_chunks}

graph_builder = StateGraph(State)

graph_builder.add_node("classify", classify_chunks)
graph_builder.add_edge(START, "classify")
graph_builder.add_edge("classify", END)

graph = graph_builder.compile()


In [ ]:

from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass


In [ ]:
def classify_and_print(chunks: List[str]):
    result = graph.invoke({"chunks": chunks, "legal_chunks": []})
    print("Legal Chunks:")
    for chunk in result["legal_chunks"]:
        print("- ", chunk)


In [ ]:
from flow import execute
# Example usage:
example_chunks = execute()

In [ ]:
print(len(example_chunks))

In [ ]:
print(example_chunks[0])

In [ ]:

classify_and_print(example_chunks)

# Example with user input:

# user_input_chunks = []

# while True:
#     try:
#         user_input = input("Enter a chunk of text (or 'done' to classify): ")
#         if user_input.lower() == "done":
#             break
#         user_input_chunks.append(user_input)
#     except:
#         break

# if user_input_chunks:
#     classify_and_print(user_input_chunks)
# else:
#     print("No chunks entered.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")

def llm_is_legal_chunk(chunk):
    """
    Uses an LLM to determine if a chunk is "legal" based on its understanding.
    This relies entirely on the LLM's interpretation of what constitutes a valid chunk.
    """
    prompt = f"""Carefully evaluate the following text. Determine whether it contains any information that is 
    meaningful, valid, and legally relevant. Assess rigorously for clarity, completeness, legal relevance, and 
    correctness. If the text contains any legal content that could be used in a legal, contractual, or 
    compliance context, respond with "Legal". If it does not contain such content or is irrelevant, 
    incomplete, vague, or nonsensical, respond with "Illegal". Do not provide any explanation—respond with 
    only "Legal" or "Illegal". Only give back the top 5.

    Text:

    {chunk}"""

    try:
        response = llm.invoke(prompt).content.lower()
        return "legal" in response
    except Exception as e:
        print(f"Error checking chunk with LLM: {e}")
        return False # Default to illegal on error

chunks = example_chunks

legal_chunks = []

for chunk in chunks:
    if llm_is_legal_chunk(chunk):
        legal_chunks.append(chunk)
        # print(chunk)

print("\nLegal Chunks (LLM-determined):")

for chunk in legal_chunks:
  print (chunk)
  print("\n")